# 09 - Options Verification

This notebook demonstrates the Bachelier/SABR models and AAD Greeks for Swaptions & Caps/Floors.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
from src.derivatives.swaption import Swaption, price_swaption_bachelier
from src.derivatives.swap import InterestRateSwap
from src.curve.nelson_siegel import NelsonSiegelCurve
from src.derivatives.aad import Dual

## 1. Bachelier Swaption Pricing

In [2]:
discount_curve = NelsonSiegelCurve(0.04, 0.0, 0.0, 1.0)
swap = InterestRateSwap(notional=1e6, fixed_rate=0.04, tenor=5.0, freq=2)
swaption = Swaption(swap, expiry=2.0, option_type="payer")

pv = price_swaption_bachelier(swaption, discount_curve, vol=0.0050)
print(f"Swaption PV (Bachelier): {float(pv):.2f}")


Swaption PV (Bachelier): 7820.29


## 2. SABR Swaption Pricing

In [3]:
sabr_params = {"alpha": 0.0050, "rho": -0.2, "nu": 0.3}
pv_sabr = price_swaption_bachelier(swaption, discount_curve, sabr_params=sabr_params)
print(f"Swaption PV (SABR): {float(pv_sabr):.2f}")


Swaption PV (SABR): 7941.39


## 3. AAD Exact Vega and Delta

In [4]:
# Using Dual numbers to get exact Greeks
from src.derivatives.bachelier import bachelier_formula

fwd = Dual(0.04)
strike = Dual(0.04)
t_exp = Dual(2.0)
vol = Dual(0.0050)
df = Dual(np.exp(-0.04 * 2.0))

px = bachelier_formula(fwd, strike, t_exp, vol, df, is_call=True)
px.backward()

print(f"Option Price: {float(px):.6f}")
print(f"Delta: {fwd.adjoint:.6f}")
print(f"Vega: {vol.adjoint:.6f}")


Option Price: 0.002604
Delta: 0.461558
Vega: 0.520813
